# Init

In [ ]:
# from tqdm import tqdm
from PIL import Image
import math
import os
from glob import glob
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import patches
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML

from scipy.optimize import curve_fit
from scipy import ndimage
from zernike import RZern

In [ ]:
def get_time(file_name:str):
    return pd.to_datetime(file_name[:17],  format="%Y%m%d %H：%M：%S")

def read_tiff_to_numpy(file_path):
    """
    Read a TIFF image file and convert it to a NumPy array.

    Args:
        file_path (str): The path to the TIFF image file.

    Returns:
        np.ndarray: A NumPy array representing the TIFF image.
    """
    try:
        # Open the TIFF image using PIL
        image = Image.open(file_path)
        if image.mode != 'L':
            image = image.convert('L')
        # Convert the image to a NumPy array
        image_array = np.array(image)
        return image_array
    except Exception as e:
        print(f"Error reading the TIFF file: {e}")
        return None

def find_centroid(img):
    """
    Calculate the centroid coordinates of an object in a binary image.

    Args:
        image (np.ndarray): A binary image represented as a NumPy array.

    Returns:
        tuple: A tuple containing the (x, y) coordinates of the centroid.
    """
    # Calculate centroid
    total = np.sum(img)
    if total == 0:
        raise ValueError("Empty image - all pixel values are zero")
    # Get image dimensions
    height, width = img.shape
    # Create coordinate grids
    x, y = np.indices((width, height))
    # Calculate weighted coordinates
    x_center = int(np.sum(x * img.T) / total)
    y_center = int(np.sum(y * img.T) / total)
    
    return x_center, y_center


def find_lightest_centroid(img):
    """
    Calculate the centroid coordinates of the lightest object in a binary image.

    Args:
        image (np.ndarray): A binary image represented as a NumPy array.

    Returns:
        tuple: A tuple containing the (y, x) coordinates of the centroid.
    """
    # Calculate centroid
    total = np.sum(img)
    if total == 0:
        raise ValueError("Empty image - all pixel values are zero")
    _img = img.copy()
    max_intensity = np.max(_img)
    _img[img!=max_intensity] = 0

    return find_centroid(_img)

def cartesian_to_polar(x, y, c_x, c_y):
    """
    Convert Cartesian coordinates to polar coordinates with a custom origin.

    Args:
        x (np.ndarray): x-coordinates of the points.
        y (np.ndarray): y-coordinates of the points.
        c_x (float): x-coordinate of the custom origin.
        c_y (float): y-coordinate of the custom origin.

    Returns:
        tuple: A tuple containing the radial distances (r) and angles (theta).
    """
    dx = x - c_x
    dy = y - c_y
    r = np.sqrt(dx**2 + dy**2)
    theta = np.arctan2(dy, dx)
    return r, theta

def polar_to_cartesian(r, theta, c_x, c_y):
    """
    Convert polar coordinates to Cartesian coordinates with a custom origin.

    Args:
        r (np.ndarray): Radial distances from the custom origin.
        theta (np.ndarray): Angles in radians.
        c_x (float): x-coordinate of the custom origin.
        c_y (float): y-coordinate of the custom origin.

    Returns:
        tuple: A tuple containing the x-coordinates (x) and y-coordinates (y).
    """
    x = c_x + r * np.cos(theta)
    y = c_y + r * np.sin(theta)
    return x, y

def normalize_data(data):
    """
    Normalize the data to the range [0, 1].
    Args:
        data (np.ndarray): The input data array.
    Returns:
        np.ndarray: The normalized data array.
    """
    min_val = np.min(data)
    max_val = np.max(data)
    if max_val == min_val:
        return data
    normalized_data = (data - min_val) / (max_val - min_val)
    return normalized_data

def gaussian(x, mu, sigma, A, b):
    """
    Define the Gaussian function.

    Args:
        x (np.ndarray): Input x values.
        A (float): Amplitude of the Gaussian.
        mu (float): Mean of the Gaussian.
        sigma (float): Standard deviation of the Gaussian.

    Returns:
        np.ndarray: Output values of the Gaussian function.
    """
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) + b


def fitting_gaussian(data):
    """
    Fit a Gaussian function to a given data series.
    Args:
        data (np.ndarray): The data series to fit a Gaussian function.
    Returns:
        tuple: A tuple containing the fitted parameters (A, b, mu, sigma) and the fitted curve.
    """
    x_data = np.arange(len(data))
    initial_guess = [np.argmax(data), 10, np.max(data), 0]
    (mu, sigma, A, b), covariance = curve_fit(gaussian, x_data, data, p0=initial_guess)

    return (mu, sigma, A, b), covariance


def calculate_diameter(sigma):
    diameter = 2 * sigma
    return diameter

def calculate_xy_diameters(image, centroid):
    """
    Calculate the diameters at y = 1/e + b in x and y directions.

    Args:
        image (np.ndarray): The input image array.
        centroid (tuple): The (y, x) coordinates of the centroid.

    Returns:
        tuple: A tuple containing the x-direction diameter and y-direction diameter.
    """
    c_y, c_x = centroid
    # Extract data for x and y directions
    y_data = image[:, c_x]
    x_data = image[c_y, :]

    # Calculate diameters
    x_diameter = calculate_diameter(x_data)
    y_diameter = calculate_diameter(y_data)

    return x_diameter, y_diameter

def extract_radial_data(image, centroid_x, centroid_y, angle):
    """
    Extract data along a radial line from the centroid at a given angle.

    Args:
        image (np.ndarray): The input image array.
        centroid_x (int): The x-coordinate of the centroid.
        centroid_y (int): The y-coordinate of the centroid.
        angle (float): The angle in degrees.

    Returns:
        np.ndarray: The extracted data along the radial line.
    """
    height, width = image.shape
    angle_rad = np.deg2rad(angle)
    max_length = int(max(
        math.sqrt(centroid_x**2 + centroid_y**2),
        math.sqrt((width - centroid_x)**2 + centroid_y**2),
        math.sqrt(centroid_x**2 + (height - centroid_y)**2),
        math.sqrt((width - centroid_x)**2 + (height - centroid_y)**2)
    ))
    distances = np.arange(-max_length, max_length + 1)
    x_coords = np.round(centroid_x + distances * np.cos(angle_rad)).astype(int)
    y_coords = np.round(centroid_y + distances * np.sin(angle_rad)).astype(int)
    valid_mask = (0 <= x_coords) & (x_coords < width) & (0 <= y_coords) & (y_coords < height)
    x_coords = x_coords[valid_mask]
    y_coords = y_coords[valid_mask]
    return image[y_coords, x_coords]


def calculate_diameter_at_angle(image, centroid_x, centroid_y, angle):
    """
    Calculate the diameter at y = 1/e + b at a given angle.

    Args:
        image (np.ndarray): The input image array.
        centroid (tuple): The (y, x) coordinates of the centroid.
        angle (float): The angle in degrees.

    Returns:
        float: The calculated diameter, or None if fitting fails or A <= 0.
    """
    radial_data = extract_radial_data(image, centroid_x, centroid_y, angle)
    (mu, sigma, A, b), conv = fitting_gaussian(radial_data)
    return calculate_diameter(sigma)

def rms(img, threshold = 0.2):
    img_blur = cv2.GaussianBlur(img, (3, 3), 100)
    img_blur = (img_blur-np.min(img_blur)) / (np.max(img_blur)-np.min(img_blur))
    mask = np.where(img_blur > threshold, 1, 0)
    blured_image = cv2.blur(img_blur,(100,100))
    mean_img = np.mean(blured_image) * mask
    sqrt_error_img = (mean_img-blured_image)**2 * mask

    return np.mean(sqrt_error_img)

In [ ]:
# --- 1. 定义Zernike多项式和拟合函数 ---

def cart2pol(x, y):
    """直角坐标转极坐标"""
    rho = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return rho, phi

def zernike_radial_poly(m, n, rho):
    """计算Zernike径向多项式 R_m^n(rho)"""
    if (n - m) % 2:  # 如果 (n-m) 是奇数，则该多项式为0
        return np.zeros_like(rho)
    if abs(m) > n:
        return np.zeros_like(rho)

    poly = np.zeros_like(rho)
    for k in range(0, (n - abs(m)) // 2 + 1):
        coef = ((-1) ** k * math.factorial(n - k)) / (
            math.factorial(k) *
            math.factorial((n + abs(m)) // 2 - k) *
            math.factorial((n - abs(m)) // 2 - k)
        )
        poly += coef * (rho ** (n - 2 * k))
    return poly

def zernike_poly(m, n, rho, phi):
    """计算Zernike多项式 Z_m^n(rho, phi)"""
    R = zernike_radial_poly(m, n, rho)
    if m >= 0:
        return R * np.cos(m * phi)
    else:
        return R * np.sin(-m * phi)

def get_zernike_indices(j):
    """根据Noll索引 j (1-based) 计算对应的 (n, m)"""
    # 使用预计算的查找表，适用于前15个Zernike模式
    noll_to_nm = {
        1: (0, 0), 2: (1, 1), 3: (1, -1), 4: (2, 0),
        5: (2, -2), 6: (2, 2), 7: (3, -1), 8: (3, 1),
        9: (3, -3), 10: (3, 3), 11: (4, 0), 12: (4, 2),
        13: (4, -2), 14: (4, 4), 15: (4, -4)
    }
    return noll_to_nm.get(j, (0, 0)) # 默认返回Piston

def fit_zernike(image, pupil_mask, m_max=4):
    """
    使用最小二乘法将图像拟合到Zernike多项式。
    :param image: 2D numpy array, 输入图像
    :param pupil_mask: 2D numpy boolean array, 有效区域掩码
    :param m_max: int, 最大拟合阶数 (Noll索引)
    :return: coefficients, zernike_modes
    """
    height, width = image.shape
    Y, X = np.mgrid[:height, :width]
    X = (X - (width - 1) / 2) / ((width - 1) / 2) # 归一化到 [-1, 1]
    Y = (Y - (height - 1) / 2) / ((height - 1) / 2) # 归一化到 [-1, 1]

    rho, phi = cart2pol(X, Y)

    # 仅在单位圆内和掩码区域内进行拟合
    valid_indices = (pupil_mask == 1) & (rho <= 1)
    valid_rho = rho[valid_indices]
    valid_phi = phi[valid_indices]
    valid_image = image[valid_indices]

    # 构建设计矩阵 (A matrix)
    num_points = np.sum(valid_indices)
    num_modes = m_max
    A = np.zeros((num_points, num_modes))

    for j in range(1, num_modes + 1):
        n, m = get_zernike_indices(j)
        # Zernike多项式在单位圆上是正交的，但通常不标准正交。
        # 为了简单起见，我们直接使用多项式值。
        # 更精确的方法是使用标准正交基或预先计算归一化系数。
        Z = zernike_poly(m, n, valid_rho, valid_phi)
        A[:, j-1] = Z

    # 使用最小二乘法求解系数 coefficients = (A^T * A)^{-1} * A^T * b
    # 或者使用更稳健的 np.linalg.lstsq
    try:
        coeffs, residuals, rank, s = np.linalg.lstsq(A, valid_image, rcond=None)
    except np.linalg.LinAlgError:
        print("拟合过程中出现线性代数错误，返回零系数。")
        coeffs = np.zeros(num_modes)

    # 重构拟合图像
    fitted_image = np.zeros_like(image, dtype=float)
    zernike_modes = {}
    for j in range(1, num_modes + 1):
        n, m = get_zernike_indices(j)
        Z_full = zernike_poly(m, n, rho, phi)
        fitted_image += coeffs[j-1] * Z_full
        zernike_modes[j] = {'coeff': coeffs[j-1], 'n': n, 'm': m, 'name': ZERNIKE_NAME_MAP.get(j, f"Z^{n}_{{{m}}}")}

    fitted_image[~valid_indices] = np.nan # 非有效区域设为NaN用于可视化

    return coeffs, fitted_image, zernike_modes

# --- 2. 定义Zernike模式名称 ---
ZERNIKE_NAME_MAP = {
    1: "Piston",
    2: "Tilt X",
    3: "Tilt Y",
    4: "Defocus",
    5: "Astigmatism 45°",
    6: "Astigmatism 0°",
    7: "Coma Y",
    8: "Coma X",
    9: "Trefoil Y",
    10: "Trefoil X",
    11: "Primary Spherical",
    12: "Secondary Astigmatism",
    13: "Secondary Astigmatism",
    14: "Tetrafoil 0°",
    15: "Tetrafoil 45°"
}

In [ ]:
# from functools import wraps

# feat_func_dict = {}

# def register(func):
#     @wraps
#     def func_wraper(**args):
        


def get_shape_center(img):
    return ndimage.center_of_mass(np.where(img>0,1,0))[::-1]

def center_spot(image, cx, cy):
    """
    计算光斑的加权质心，并将图像和掩码平移，使质心与图像中心对齐。
    :param image: 2D numpy array, 输入图像 (通常是滤波后的)
    :param initial_mask: 2D numpy boolean array, 初步提取的光斑掩码
    :return: centered_image, centered_mask, (dx, dy)
    """
    h, w = image.shape
    center_x, center_y = w / 2.0, h / 2.0
    dx = center_x - cx
    dy = center_y - cy
    centered_image = ndimage.shift(image, (dy, dx), order=1, cval=0, prefilter=False)

    return centered_image, (dx, dy)

def preprocess(img):
    int_max = np.max(img)
    threshold = 0.02 * int_max
    img = img - threshold
    img = np.where(img<0, 0, img) / int_max
    img = cv2.blur(img,(100,100))
    cx, cy = get_shape_center(img)
    img, _ = center_spot(img, cx, cy)
    return img

def get_brightness_center(img):
    _img = np.where(img==np.max(img), 1, 0)
    return get_mass_center(_img)

def get_mass_center(img):
    return ndimage.center_of_mass(img)[::-1]

def zernike_tile(img):
    mask = np.where(img < 0, 0, 1)
    coefficients, fitted_image, zernike_modes = fit_zernike(img, mask, m_max=3)
    return coefficients[1], coefficients[2]

def intense_diff(img):
    h, w = img.shape[0], img.shape[1]
    quadrants = [np.sum(img[h//2:,w//2:]), np.sum(img[:h//2,w//2:]), np.sum(img[:h//2,:w//2]), np.sum(img[h//2:,:w//2])]
    return quadrants[0]+quadrants[1]-quadrants[2]-quadrants[3], quadrants[0]-quadrants[1]-quadrants[2]+quadrants[3]

In [ ]:
import cv2

def spot_border(image):
    """
    处理光斑图片，计算噪声阈值，去除噪声并拟合包含光斑的圆形。

    参数:
    image (numpy.ndarray): 输入的光斑图片，应为单通道灰度图像。

    返回:
    numpy.ndarray: 去除噪声后的图像。
    tuple: 拟合圆形的圆心坐标 (x, y) 和半径。
    """
    noise_threshold = np.max(image.astype(np.uint8)) * (1/math.e)
    denoised_image = np.where(image > noise_threshold, image, 0)
    near_binary_img = cv2.threshold(denoised_image, noise_threshold, 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(near_binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        ((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
        center = (int(x), int(y))
        radius = int(radius)
    else:
        center = (0, 0)
        radius = 0

    return denoised_image, (center, radius)

def find_spot_border(image):
    """
    处理光斑图片，计算噪声阈值，去除噪声并拟合包含光斑的圆形。

    参数:
    image (numpy.ndarray): 输入的光斑图片，应为单通道灰度图像。

    返回:
    numpy.ndarray: 去除噪声后的图像。
    tuple: 拟合圆形的圆心坐标 (x, y) 和半径。
    """
    # 步骤 1: 高斯降噪
    denoised_image = cv2.GaussianBlur(image, (3, 3), 0)
    
    # 步骤 2: canny边缘检测
    noise_threshold = np.max(denoised_image) * (1/math.e)
    denoised_image = np.where(denoised_image > noise_threshold, denoised_image, 0)
    # 步骤 3: 去除噪声
    denoised_image = cv2.fastNlMeansDenoising(denoised_image, None, 10, 7, 21)
    # denoised_image = cv2.Canny(image, 1, 1)
    # 步骤 4: 二值化
    near_binary_img = cv2.threshold(denoised_image, noise_threshold, 255, cv2.THRESH_BINARY)[1]
    # 步骤 5: 拟合一个圆形正好包含光斑
    contours, _ = cv2.findContours(near_binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # 找到最大的轮廓
        largest_contour = max(contours, key=cv2.contourArea)
        ((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
        center = (int(x), int(y))
        radius = int(radius)
    else:
        center = (0, 0)
        radius = 0

    return denoised_image, (center, radius)

def ellipse_fit(image):
    noise_threshhold = np.max(image)*0.3
    binary_image = cv2.threshold(image, noise_threshhold, 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # 找到最大的轮廓
        largest_contour = max(contours, key=cv2.contourArea)
        (ellipse_center_x, ellipse_center_y),(short_axis, long_axis),angle = cv2.fitEllipse(largest_contour)
        return (ellipse_center_x, ellipse_center_y),(short_axis, long_axis),angle

In [ ]:
# display funcs

def display_3d_image(image_array):
    # Generate x, y coordinates
    x = np.arange(image_array.shape[1])
    y = np.arange(image_array.shape[0])
    X, Y = np.meshgrid(x, y)

    # Create a 3D surface plot
    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=image_array)])

    # Update layout
    fig.update_layout(
        title='3D Visualization of Far Spot Image',
        scene=dict(
            xaxis_title='X Coordinate',
            yaxis_title='Y Coordinate',
            zaxis_title='Brightness'
        )
    )

    HTML(fig.to_html())
    
def display_image(image_array, c_x, c_y):
    fig = px.imshow(image_array, color_continuous_scale='gray')
    # Add the centroid as a red spot
    fig.add_trace(go.Scatter(
        x=[c_x],
        y=[c_y],
        mode='markers',
        marker=dict(color='red', size=10),
        name='Centroid'
    ))
    fig.update_layout(title='Image with Centroid Marked')
    fig.show()

# Load data

In [ ]:
beam_axis_img_dir = Path('../data/experiment_data/20250923/001/digitaloptical4Floor/光轴image/20250923 16：09：19(001-1%平顶)')
files = list(beam_axis_img_dir.glob("*.TIFF"))

axis_data = pd.DataFrame(files, columns=['path'])
axis_data['time'] = axis_data['path'].apply(lambda s: get_time(s.name))
axis_data['img'] = axis_data['path'].apply(read_tiff_to_numpy)
axis_data['brightness'] = axis_data['img'].apply(np.max)

valid = axis_data.brightness > axis_data.brightness.median()
axis_data = axis_data[valid]

far_spot_image = axis_data.img.tolist()[0]

In [ ]:
beam_pupil_img_dir = Path('../data/experiment_data/20250923/001/digitaloptical4Floor/光瞳image/20250923 16：09：19(001-1%平顶)')
files = list(beam_pupil_img_dir.glob("*.TIFF"))
pupil_data = pd.DataFrame(files, columns=['path'])
pupil_data['time'] = pupil_data['path'].apply(lambda s: get_time(s.name))
pupil_data['img'] = pupil_data['path'].apply(read_tiff_to_numpy)
pupil_data['brightness'] = pupil_data['img'].apply(np.max)

pupil_data = pupil_data[pupil_data.brightness > 10]

near_spot_image = pupil_data.img.tolist()[0]

# Single analysis

## 光轴束腰半径

In [ ]:
axis_data.brightness.median()

In [ ]:
far_spot_image = axis_data[axis_data.brightness > axis_data.brightness.median()].img.tolist()[0]

c_x, c_y = find_lightest_centroid(far_spot_image)
centroid = (c_y, c_x)


fig = px.imshow(far_spot_image, color_continuous_scale='gray')
# Add the centroid as a red spot
fig.add_trace(go.Scatter(
    x=[c_x],
    y=[c_y],
    mode='markers',
    marker=dict(color='red', size=2),
    name='Centroid'
))
fig.update_layout(title='Image with Centroid Marked')
fig.show()

拟合高斯函数：

$$
f(x) = \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)
$$

高斯函数表达式为 $ y = \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)+b $ ，要计算 $ (y = \frac{1}{e^2}+b) $ 对应的直径，我们需要先求解方程 $ (\exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)+b=\frac{1}{e}+b) $ ，化简该方程可得：

$$
\begin{align*} 
\exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)&=\frac{1}{e^2}\\
-\frac{(x - \mu)^2}{2\sigma^2}&=\ln\left(\frac{1}{e^2}\right)= -2\\
(x - \mu)^2&=2\sigma^2( 2)\\
x&=\mu\pm{2\sigma} \end{align*}
$$

直径 $ (d) $ 就是两个解的差值，即 $ (d = 2\sqrt{2\sigma^2(2)}) = 4 \sigma $ 。


In [ ]:
for degree in range(0, 180, 45):
    y_data = extract_radial_data(far_spot_image, c_x, c_y, degree)
    y_data = normalize_data(y_data)

    (mu, sigma), covariance = fitting_gaussian(y_data)
    diameter = 4* sigma
    print(f"{degree=} Diameter: {diameter*22:.2f}um {(mu, sigma)=}")

## 椭圆拟合

* 椭圆质心 x,y
* 轴长
* 角度

In [ ]:
# TODO: 环围能量计算阈值

noise_threshhold = np.max(far_spot_image)*0.3
noise_threshhold

In [ ]:
binary_image = cv2.threshold(far_spot_image, noise_threshhold, 255, cv2.THRESH_BINARY)[1]
plt.imshow(binary_image)

In [ ]:
contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if contours:
    # 找到最大的轮廓
    largest_contour = max(contours, key=cv2.contourArea)
    print(cv2.fitEllipse(largest_contour))

## 远场光斑

### 降噪及边缘

在图像处理领域，模糊是一种常见的操作，用于减少图像中的噪声或细节，使图像看起来更加平滑。OpenCV提供了多种模糊处理方法，包括均值模糊、中值模糊和高斯模糊。

1. 均值模糊
    
    均值模糊是最简单的一种模糊方法，它通过取卷积核覆盖区域内所有像素的平均值来代替中心像素值。均值模糊的代码如下：

```[python]
import cv2 as cv
import numpy as np

def blur_demo(image):
dst = cv.blur(image, (5, 5)) # 卷积核大小为5x5
cv.imshow("blur_demo", dst)

src = cv.imread('pic.jpg')
cv.imshow("org", src)
blur_demo(src)
cv.waitKey(0)
cv.destroyAllWindows()
```

2. 中值模糊

    中值模糊通过取卷积核覆盖区域内所有像素的中值来代替中心像素值，特别适用于去除椒盐噪声。中值模糊的代码如下：
```[python]
import cv2 as cv
import numpy as np

def med_blur_demo(image):
dst = cv.medianBlur(image, 5) # 卷积核大小为5
cv.imshow("med_blur_demo", dst)

src = cv.imread('saltImage.jpg')
cv.imshow("org", src)
med_blur_demo(src)
cv.waitKey(0)
cv.destroyAllWindows()
```
3.高斯模糊

    高斯模糊使用高斯核函数对图像进行平滑处理，能够有效去除高斯噪声。高斯模糊的代码如下：
```[python]
import cv2 as cv
import numpy as np

def gauss_blur_demo(image):
dst = cv.GaussianBlur(image, (5, 5), 0) # 卷积核大小为5x5，标准差为0
cv.imshow("gauss_blur_demo", dst)

src = cv.imread('pic.jpg')
cv.imshow("org", src)
gauss_blur_demo(src)
cv.waitKey(0)
cv.destroyAllWindows()
```

In [ ]:
root_dir = "/home/tifo/workspace/data/daily_data/"

file_list = glob(root_dir+"20250819/20250819001/digitaloptical4Floor/光瞳image/20250819 10：04：19(100%平顶31)/*.TIFF")

near_spot_image = read_tiff_to_numpy(file_list[4])

c_x, c_y = find_centroid(near_spot_image)
display_image(near_spot_image, c_x, c_y)


** 计算梯度： **
* 水平方向梯度： sobelx = cv2.Sobel(img_blur, cv2.CV_64F, 1, 0, ksize=3) 
* 垂直方向梯度： sobely = cv2.Sobel(img_blur, cv2.CV_64F, 0, 1, ksize=3)
* 合并梯度： sobel_combined = cv2.addWeighted(cv2.convertScaleAbs(sobelx), 0.5, cv2.convertScaleAbs(sobely), 0.5, 0)

In [ ]:
# 应用高斯模糊（减少噪声）
img_blur = cv2.GaussianBlur(near_spot_image, (3, 3), 100)

sobelx = cv2.Sobel(img_blur, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_blur, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.addWeighted(cv2.convertScaleAbs(sobelx), 0.5, cv2.convertScaleAbs(sobely), 0.5, 0)
plt.imshow(sobel_combined)

In [ ]:
near_spot_image = cv2.GaussianBlur(near_spot_image, (3, 3), 0)

threshold = np.max(near_spot_image) * (1/math.e)
threshold

In [ ]:
near_binary_img = cv2.threshold(near_spot_image, threshold, 255, cv2.THRESH_BINARY)[1]
plt.imshow(near_binary_img, cmap='gray')

In [ ]:
contours, _ = cv2.findContours(near_binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
largest_contour = max(contours, key=cv2.contourArea)

((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
print((x, y), radius)

fig, ax = plt.subplots(1)
ax.imshow(near_spot_image, cmap='gray')
circle = patches.Circle((x, y), radius, linewidth=2, edgecolor='r', facecolor='none')
ax.add_patch(circle)
ax.set_title('Denoised Image with Fitted Circle')
plt.show()

In [ ]:
near_spot_image = cv2.GaussianBlur(near_spot_image, (3, 3), 0)
noise_threshold = np.mean(near_spot_image) * 0.5

noise_threshold

In [ ]:
denoised_image, (center, radius) = find_spot_border(near_spot_image)
print(center, radius)
fig, ax = plt.subplots(1)
ax.imshow(near_spot_image, cmap='gray')
circle = patches.Circle(center, radius, linewidth=2, edgecolor='r', facecolor='none')
ax.add_patch(circle)
ax.set_title('Denoised Image with Fitted Circle')
plt.show()


### 光斑均匀度

1. 列项积分
2. RMS
3. zernike分解

#### data test

In [ ]:
%%time
data = {}

for i, (dir, (o_x,o_y)) in enumerate(samples):
    imgs_path = dir.glob('*.TIFF')
    label = f'{(o_x,o_y)}'
    data[label] = data.get(label, list())
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        img = preprocess(img)
        x,y = intense_diff(img)
        data[label].append((x,y))
        
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(x=sample[:,0], y=sample[:,1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(center[0],center[1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
%%time
def shape_mass_center_diff(img:np.ndarray) -> tuple[float,float]:
    threshold = 0.02 * np.max(img)
    img = img - threshold
    mask = np.where(img<0, 0, 1)
    img = np.where(img<0, 0, img)
    sc_x, sc_y = find_centroid(mask)
    mc_x, mc_y = find_centroid(img)
    
    return mc_x-sc_x, mc_y-sc_y

data = {}

for i, (dir, (o_x,o_y)) in enumerate(samples):
    imgs_path = dir.glob('*.TIFF')
    label = f'{(o_x,o_y)}'
    data[label] = data.get(label, list())
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        x,y = shape_mass_center_diff(img)
        data[label].append((x,y))
        
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(x=sample[:,0], y=sample[:,1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
%%time
dir, (cx, cy) = samples[1]
centers = []
def get_center(img_path):
    img = read_tiff_to_numpy(img_path)
    threshold = 0.02 * np.max(img)
    img = img - threshold
    img = np.where(img<0, 0, img)
    return find_centroid(img)

for img_path in dir.glob('*.TIFF'):
    centers.append(get_center(img_path))

centers = np.array(centers)
Center = np.mean(centers, axis=0)

def mass_center_diff(img:np.ndarray) -> tuple[float,float]:
    threshold = 0.02 * np.max(img)
    img = img - threshold
    mask = np.where(img>0, 1, 0)
    img = np.where(img<0, 0, img)
    sc_x, sc_y = Center
    mc_x, mc_y = find_centroid(img)
    
    return mc_x-sc_x, mc_y-sc_y

data = {}

for i, (dir, (o_x,o_y)) in enumerate(samples):
    imgs_path = dir.glob('*.TIFF')
    label = (o_x,o_y)
    data[label] = data.get(label, list())
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        x,y = mass_center_diff(img)
        data[label].append((x,y))
        
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(x=sample[:,0], y=sample[:,1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(center[0],center[1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
%%time
def zernike_tile_to_fix_center(img):
    blured_image = cv2.blur(img,(100,100))
    intens_min, intens_max = np.min(blured_image), np.max(blured_image)
    threshold = 0.02 * np.max(blured_image)
    blured_image = blured_image-threshold
    mask = np.where(blured_image < 0, 0, 1)
    blured_image = np.where(blured_image < 0, 0, blured_image)
    image_array = (blured_image - intens_min) / (intens_max - intens_min)
    cx, cy = Center
    centered_image, _ = center_spot(image_array, cx, cy)
    centered_mask, (dx, dy) = center_spot(mask, cx, cy)
    coefficients, fitted_image, zernike_modes = fit_zernike(centered_image, centered_mask, m_max=3)
    return coefficients[1], coefficients[2]

data = {}

for i, (dir, (o_x,o_y)) in enumerate(samples):
    imgs_path = dir.glob('*.TIFF')
    label = f'{(o_x,o_y)}'
    data[label] = data.get(label, list())
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        x,y = zernike_tile_to_fix_center(img)
        data[label].append((x,y))
        
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(x=sample[:,0], y=sample[:,1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

In [ ]:
%%time
def zernike_tile(img):
    blured_image = cv2.blur(img,(100,100))
    intens_min, intens_max = np.min(blured_image), np.max(blured_image)
    threshold = 0.02 * np.max(blured_image)
    blured_image = blured_image-threshold
    mask = np.where(blured_image < 0, 0, 1)
    blured_image = np.where(blured_image < 0, 0, blured_image)
    image_array = (blured_image - intens_min) / (intens_max - intens_min)
    cx, cy = find_centroid(image_array)
    centered_image, _ = center_spot(image_array, cx, cy)
    centered_mask, (dx, dy) = center_spot(mask, cx, cy)
    coefficients, fitted_image, zernike_modes = fit_zernike(centered_image, centered_mask, m_max=3)
    return coefficients[1], coefficients[2]

data = {}

for i, (dir, (o_x,o_y)) in enumerate(samples):
    imgs_path = dir.glob('*.TIFF')
    label = f'{(o_x,o_y)}'
    data[label] = data.get(label, list())
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        x,y = zernike_tile(img)
        data[label].append((x,y))
        
for i, (k,v) in enumerate(data.items()):
    sample = np.array(v)
    center = np.mean(sample, axis=0)
    plt.scatter(x=sample[:,0], y=sample[:,1], c=colors[i,:])
    plt.text(center[0],center[1],k,c=colors[i,:])
plt.show()
plt.close()

#### data load

In [ ]:
root_dir = Path('/home/tifo/workspace/data/光瞳均匀度测试')
sample_dir = root_dir.glob('*(全子束弱光平顶)*')

def get_xy(path:Path):
    path_name = path.name
    xy_str = path_name.split(')')[-1]
    xy = xy_str.split('.')
    return int(xy[0])-10, int(xy[1])+130 if len(xy)==2 else 0

samples = [(dir, get_xy(dir)) for dir in sample_dir]

import tqdm

data = []

for i, (dir, (x,y)) in enumerate(tqdm.tqdm(samples)):
    imgs_path = dir.glob('*.TIFF')
    for img_path in imgs_path:
        img = read_tiff_to_numpy(img_path)
        data.append((img, x, y, img_path))

data = pd.DataFrame(data, columns=['img', 'x', 'y', 'path'])
data['tm_xy'] = data.apply(lambda s: (s['x'], s['y']), axis=1)
# data.to_pickle('uniformated_pd.pkl', compression='zip')
colors = np.random.rand(len(samples), 3)

In [ ]:
sample_df = pd.DataFrame(samples)
sample_df.columns = ['dir','coord']
sample_df

img_a_dir = sample_df[sample_df.coord==(-30,30)]['dir'].tolist()[0]
img_a = read_tiff_to_numpy(img_a_dir.glob('*.TIFF').__next__())

img_b_dir = sample_df[sample_df.coord==(0,30)]['dir'].tolist()[0]
img_b = read_tiff_to_numpy(img_b_dir.glob('*.TIFF').__next__())

fig, [ax1, ax2] = plt.subplots(1,2)
ax1.imshow(img_a)
ax2.imshow(img_b)

In [ ]:
normed_img_a = preprocess(img_a)
normed_img_b = preprocess(img_b)

display_image(normed_img_a, *get_brightness_center(normed_img_a))
display_image(normed_img_b, *get_brightness_center(normed_img_b))

#### feature test

In [ ]:
%time data['preprocess'] = data.img.apply(preprocess)

%time mass_center = data.preprocess.apply(get_mass_center)
data['mass_center_x'] = mass_center.apply(lambda s: s[0])
data['mass_center_y'] = mass_center.apply(lambda s: s[1])

%time brightness = data.preprocess.apply(get_brightness_center)
data['brightness_x'] = brightness.apply(lambda s: s[0])
data['brightness_y'] = brightness.apply(lambda s: s[1])

%time intense = data.preprocess.apply(intense_diff)
data['intense_x'] = intense.apply(lambda s: s[0])
data['intense_y'] = intense.apply(lambda s: s[1])

%time data['zernike_x'], data['zernike_y']  = zip(*data.preprocess.apply(zernike_tile))

%time data['spot_center'], data['spot_border'] = zip(*data.preprocess.apply(_find_spot_border))

%time data['rms'] = zip(*valid_pupil_data.img.apply(_find_spot_border))

data

In [ ]:
agg_prop = {
    'zernike_x': np.mean,
    'intense_x': np.mean,
    'mass_center_x': np.mean,
    'brightness_x': np.mean,
    'zernike_y': np.mean,
    'intense_y': np.mean,
    'mass_center_y': np.mean,
    'brightness_y': np.mean,
    'rms': np.mean,
}

In [ ]:
grouped_df = data.groupby('x').agg(agg_prop)

grouped_df.plot(subplots=True)

In [ ]:
grouped_df = data.groupby('y').agg(agg_prop)

grouped_df.plot(subplots=True)

#### regression model test

In [ ]:
data.columns

In [ ]:
# data.groupby('y').agg({
#     'intense_x': np.mean,
#     'mass_center_x': np.mean,
#     'brightness_x': np.mean,
#     'intense_y': np.mean,
#     'mass_center_y': np.mean,
#     'brightness_y': np.mean,
# })
feat_data, x_data, y_data = [], [], []
feat_names = ['mass_center_x', 'mass_center_y','brightness_x', 'brightness_y', 'intense_x', 'intense_y']
for i, line in data.iterrows():
    feat = line[feat_names].tolist()
    feat_data.append(feat)
    x_data.append(line['x'])
    y_data.append(line['y'])

feat_data, x_data, y_data = np.array(feat_data), np.array(x_data), np.array(y_data)

#### non-linear transform

In [ ]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(feat_data, x_data, test_size=0.2)
models, predictions = reg.fit(train_X, test_X, train_y, test_y)

# 打印性能报告
print(models)

In [ ]:
from sklearn.linear_model import LinearRegression as RG
clf = RG()
clf.fit(train_X, train_y)

pred_y = clf.predict(test_X)
plt.plot(np.arange(len(test_X)), pred_y, label='pred')
plt.plot(np.arange(len(test_X)), test_y, label='true')
plt.legend()
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor as RG
clf = RG()
clf.fit(train_X, train_y)

pred_y = clf.predict(test_X)
plt.plot(np.arange(len(test_X)), pred_y, label='pred')
plt.plot(np.arange(len(test_X)), test_y, label='true')
plt.legend()
plt.show()

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(feat_data, y_data, test_size=0.15)
models, predictions = reg.fit(train_X, test_X, train_y, test_y)

# 打印性能报告
print(models)

In [ ]:
from sklearn.linear_model import LinearRegression as RG
clf = RG()
clf.fit(train_X, train_y)

pred_y = clf.predict(test_X)
plt.plot(np.arange(len(test_X)), pred_y, label='pred')
plt.plot(np.arange(len(test_X)), test_y, label='true')
plt.legend()
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor as RG
clf = RG()
clf.fit(train_X, train_y)

pred_y = clf.predict(test_X)
plt.plot(np.arange(len(test_X)), pred_y, label='pred')
plt.plot(np.arange(len(test_X)), test_y, label='true')
plt.legend()
plt.show()

#### 泽尼克

泽尼克多项式通过两个整数索引来定义：径向阶数 **n** 和角向频率 **m**。Noll为了方便，将它们重新排列为一个单一下标 **j**。以下是低阶泽尼克系数（按Noll索引 j 排列）的物理含义，这也是上面代码输出的内容：

| j (Noll) | (n, m) | 系数值的意义 | 中文名称 | 英文名称 |
| :--- | :--- | :--- | :--- | :--- |
| 1 | (0, 0) | 代表整体的平均值或直流分量。在强度拟合中代表光斑平均亮度。 | **活塞 (平移)** | Piston |
| 2 | (1, 1) | 代表沿X轴的线性倾斜。如果系数为正，表示光斑强度在+X方向上增强。 | **X方向倾斜** | X-Tilt |
| 3 | (1, -1) | 代表沿Y轴的线性倾斜。 | **Y方向倾斜** | Y-Tilt |
| 4 | (2, 0) | 代表二次径向变化。正值表示中心亮、边缘暗；负值表示中心暗、边缘亮。 | **离焦** | Defocus |
| 5 | (2, -2) | 代表在45°和135°方向上的像散。形状类似马鞍。 | **45°像散** | Oblique Astigmatism |
| 6 | (2, 2) | 代表在0°和90°（垂直和水平）方向上的像散。 | **0°像散** | Vertical Astigmatism |
| 7 | (3, -1) | 描述一种彗星状的畸变，一侧比另一侧更亮/更暗，沿Y轴分布。 | **垂直彗差** | Vertical Coma |
| 8 | (3, 1) | 沿X轴分布的彗差。 | **水平彗差** | Horizontal Coma |
| 9 | (3, -3) | 描述三叶草形状的强度分布，沿Y轴对称。 | **垂直三叶草** | Vertical Trefoil |
| 10 | (3, 3) | 沿X轴对称的三叶草。 | **倾斜三叶草** | Oblique Trefoil |
| 11 | (4, 0) | 代表更高阶的径向变化，与球差相关。正值会使边缘强度相对于中心进一步降低。 | **球差** | Spherical Aberration |

**总结一下**：

  * **系数的绝对值大小**：表示该项像差或形状特征的强度。绝对值越大，该特征越显著。
  * **系数的符号**：表示该特征的方向。例如，倾斜（Tilt）系数的正负号决定了倾斜的方向。

对于您提到的**干涉条纹**，它们是高频信息。在泽尼克拟合中，这些高频成分会被更高阶的泽尼克项来近似，或者在阶数不够高时，它们会成为拟合的**残差**（Residual），即原始图像与拟合图像的差异。通过观察残差图，可以判断拟合效果以及哪些特征（如干涉条纹）没有被模型很好地捕捉。

In [ ]:
threshold = 0.2 * np.max(near_spot_image)
mask = np.zeros_like(near_spot_image)
mask[near_spot_image > threshold]=1
# print(f'{threshold=}')
# img_blur[img_blur < threshold] = 0
blured_image = cv2.blur(img_blur,(100,100))
plt.imshow(blured_image * mask, cmap='gray')

In [ ]:
%%time
# 强度归一化，将图像值映射到[0, 1]范围
target_img = near_spot_image
intens_min, intens_max = np.min(target_img), np.max(target_img)
image_array = (target_img - intens_min) / (intens_max - intens_min)
 # --- 定位并移动光斑到中心 ---
print("\n--- 正在将光斑移动到图像中心 ---")
cx, cy = find_centroid(np.ones_like(blured_image)*mask)
centered_image, _ = center_spot(image_array, cx, cy)
centered_mask, (dx, dy) = center_spot(mask, cx, cy)
print("-----------------------------------\n")

# 可视化居中后的图像和掩码
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(centered_image, cmap='gray')
axs[0].set_title('centered image')
axs[0].axis('off')

axs[1].imshow(centered_mask, cmap='gray')
axs[1].set_title('centered mask')
axs[1].axis('off')

# 对居中后的图像再次应用高斯滤波（可选，但保持一致性）
centered_image_filtered = ndimage.maximum_filter(image_array,(15,15),mode='nearest')
axs[2].imshow(centered_image_filtered, cmap='gray')
axs[2].set_title(f'centered gaussian image: {centered_image_filtered.shape[::-1]}')
axs[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
%%time
# --- 在居中后的图像上执行Zernike拟合 ---
M = 11 # 拟合到11阶（包含前11个Zernike模式）
pupil_mask = centered_mask[centered_mask==1]
coefficients, fitted_image, zernike_modes = fit_zernike(centered_image_filtered, centered_mask, m_max=M)
# 可视化最终拟合结果
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(centered_image_filtered, cmap='gray')
axs[0].set_title('input image')
axs[0].axis('off')

axs[1].imshow(fitted_image, cmap='RdBu')
axs[1].set_title('Zernike fitting result')
axs[1].axis('off')

residual = centered_image_filtered - fitted_image
im_residual = axs[2].imshow(residual, cmap='seismic')
axs[2].set_title('residual(origin-fitted)')
axs[2].axis('off')
plt.colorbar(im_residual, ax=axs[2], shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
coefficients

#### RMS

In [ ]:
# TODO: 用聚类来找出边缘
img_blur = cv2.GaussianBlur(near_spot_image, (3, 3), 100)
img_blur = (img_blur-np.min(img_blur)) / (np.max(img_blur)-np.min(img_blur))

plt.imshow(img_blur, cmap='gray')

In [ ]:
threshold = 0.2 * np.max(near_spot_image)
mask = np.zeros_like(near_spot_image)
mask[near_spot_image > threshold]=1
# print(f'{threshold=}')
# img_blur[img_blur < threshold] = 0
blured_image = cv2.blur(img_blur,(100,100))
plt.imshow(blured_image * mask, cmap='gray')

In [ ]:
mean_img = np.ones_like(mask) * np.mean(blured_image) * mask
sqrt_error_img = (mean_img-blured_image)**2 * mask
plt.imshow(sqrt_error_img)
plt.title(f'mse={np.mean(sqrt_error_img)}')
plt.show()

#### 质心和形心的偏差

In [ ]:
mc = find_centroid(blured_image)
display_image(blured_image, *mc)

In [ ]:
sc = find_centroid(np.ones_like(blured_image)*mask)
display_image(blured_image, *sc)

In [ ]:
(mc, sc)

# Sequential analysis

In [ ]:
%time axis_data['ellipse_center'], axis_data['axises'], axis_data['axis_angle'] = zip(*axis_data['img'].apply(ellipse_fit))
%time axis_data['diameter'] = axis_data.apply(lambda s: calculate_diameter_at_angle(s['img'], *s.ellipse_center, s.axis_angle), axis=1)

axis_data

In [ ]:
def _find_spot_border(img):
    _, (center, r) = find_spot_border(img)
    return center, r

%time valid_pupil_data['center_mass'] = valid_pupil_data.img.apply(find_centroid)
%time valid_pupil_data['center_shape'], valid_pupil_data['spot_border'] = zip(*valid_pupil_data.img.apply(_find_spot_border))
%time valid_pupil_data['rms'] = valid_pupil_data.img.apply(rms)

valid_pupil_data

## centroid shifting

In [ ]:
# Create a figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 15), sharex=True)
# Plot distance from origin
axes[0].plot(data['time'], data['distance'], label='Distance from (0, 0)', color='blue')
axes[0].set_ylabel('Distance')
axes[0].set_title('Distance from Origin Over Time')
axes[0].legend()
axes[0].grid(True)

# Plot x-coordinate changes
axes[1].plot(data['time'], data['x'], label='X-coordinate', color='green')
axes[1].set_ylabel('X-coordinate')
axes[1].set_title('X-coordinate Over Time')
axes[1].legend()
axes[1].grid(True)

# Plot y-coordinate changes
axes[2].plot(data['time'], data['y'], label='Y-coordinate', color='red')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Y-coordinate')
axes[2].set_title('Y-coordinate Over Time')
axes[2].legend()
axes[2].grid(True)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust the layout
plt.tight_layout()
plt.show()


## light flow

In [ ]:
image_list = valid_pupil_data.img.tolist()
len(image_list)

In [ ]:
color = np.random.randint(0,255,(100,3))

feature_params = dict(
    maxCorners = 100,
    qualityLevel = 0.3,
    minDistance = 2
)

p0 = cv2.goodFeaturesToTrack(image_list[0][0], mask=None, **feature_params)
mask = np.zeros_like(image_list[0][0])

lk_params = dict(
    winSize=(5,5),
    maxLevel=2
)

for image, file_name in image_list:
    p1, st, err = cv2.calcOpticalFlowPyrLK(image_list[0][0], image, p0, None, **lk_params)
    good_new = p1[st==1]
    good_old = p0[st==1]

    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a,b = new.ravel().astype(int)
        c,d = old.ravel().astype(int)

        mask = cv2.line(mask, (a,b), (c,d), 

## Brightness

In [ ]:
spot_files = glob(r'data\0611\1F\光轴image\20250611 15：52：11(60%（31束）)\*.TIFF')
len(spot_files)

In [ ]:
def get_highest_bright(file_path):
    img = read_tiff_to_numpy(file_path)
    return np.max(img)

plt.plot([get_highest_bright(file) for file in spot_files])

In [ ]:
# load data

data = pd.read_csv('./4f分析/光轴质心20250606 10：23：07(20%平顶31).TXT', encoding='gbk', header=None, sep='\t')
data.columns = ['time', 'x', 'y']

data['time'] = pd.to_datetime(data['time'], format='%Y-%m-%d-%H：%M：%S.%f')
data['distance'] = np.sqrt(data['x']**2 + data['y']**2)

In [ ]:
# Create a figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 15), sharex=True)
# Plot distance from origin
axes[0].plot(data['time'], data['distance'], label='Distance from (0, 0)', color='blue')
axes[0].set_ylabel('Distance')
axes[0].set_title('Distance from Origin Over Time')
axes[0].legend()
axes[0].grid(True)

# Plot x-coordinate changes
axes[1].plot(data['time'], data['x'], label='X-coordinate', color='green')
axes[1].set_ylabel('X-coordinate')
axes[1].set_title('X-coordinate Over Time')
axes[1].legend()
axes[1].grid(True)

# Plot y-coordinate changes
axes[2].plot(data['time'], data['y'], label='Y-coordinate', color='red')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Y-coordinate')
axes[2].set_title('Y-coordinate Over Time')
axes[2].legend()
axes[2].grid(True)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust the layout
plt.tight_layout()
plt.show()


In [ ]:
EXP_NAME = '20250611 15：55：47(80%（31束）)'

In [ ]:
far_spot_res = pd.read_pickle(f'data/20250611/1F/光轴image/{EXP_NAME}/{EXP_NAME}.pkl')
far_spot_res['time'] = pd.to_datetime(far_spot_res['time'])
if 'Unnamed: 0' in far_spot_res.columns:
    far_spot_res.drop(columns=['Unnamed: 0'], inplace=True)
    
far_spot_res.info()
far_spot_res.set_index('time', inplace=True)

    
far_spot_res

In [ ]:
target_time_series = far_spot_res.index

def generate_time_sequence(start: datetime, end: datetime):
    """
    生成从起始时间到结束时间的时间序列，相邻时间间隔0.001秒
    
    Args:
        start: 起始时间点（包含）
        end: 结束时间点（包含）
    
    Returns:
        时间序列列表
    """
    if start > end:
        raise ValueError("结束时间必须大于等于起始时间")
    
    time_sequence = []
    current_time = start
    while current_time <= end:
        time_sequence.append(current_time)
        # 每次递增0.01秒（10毫秒）
        current_time += timedelta(milliseconds=10)
    
    return time_sequence

target_time_series = generate_time_sequence(target_time_series[0], target_time_series[-1])
time_index = pd.DataFrame(target_time_series, columns=['time'])

time_index

In [ ]:
far_spot_selected = pd.merge(time_index, far_spot_res, on='time', how='outer')
far_spot_selected[['centroid_x', 'centroid_y']].interpolate(limit_direction='both', method='pchip', inplace=True)
far_spot_selected.interpolate(inplace=True, method='linear')

far_spot_selected.set_index('time', inplace=True)

far_spot_selected = pd.merge(time_index, far_spot_selected, on='time', how='left')
far_spot_selected.set_index('time', inplace=True)
far_spot_selected

In [ ]:
tm_res = pd.read_csv(f'data/20250611/1F/快反镜数据/{EXP_NAME}.TXT', encoding='gbk', sep='\t', header=None)

tm_res.columns = ['time', 'tm_x', 'tm_y']
tm_res['time'] = pd.to_datetime(tm_res.time, format='%Y-%m-%d-%H：%M：%S.%f')
tm_res.info()

tm_res.set_index('time', inplace=True)
tm_res

In [ ]:
tm_selected = pd.merge(time_index, tm_res, on='time', how='outer')
tm_selected[['tm_x','tm_y']].interpolate(method='pchip', limit_direction='both', inplace=True)
tm_selected = pd.merge(time_index, tm_selected, on='time', how='left')
tm_selected.set_index('time')
tm_selected

In [ ]:
merged = pd.merge(far_spot_selected, tm_selected, on='time', how='outer')
merged.to_csv(f'{EXP_NAME}-interpolated.csv', index=False)

merged

In [ ]:
# 

centroid_x = far_spot_selected['centroid_x'].to_numpy()
dt = far_spot_selected['ms_diff'].to_numpy()

n = len(centroid_x)
fft_vals = np.fft.fft(centroid_x - np.mean(centroid_x))  # 去除直流分量
fft_amp = np.abs(fft_vals) / n  # 计算振幅谱（归一化）

# 计算频率轴（单位：Hz）
dt_avg = np.mean(dt) / 1000  # 平均采样间隔（秒）
freq = np.fft.fftfreq(n, d=dt_avg)[:n//2]  # 取正频率部分

# 绘制频谱图
plt.figure()
plt.plot(freq, fft_amp[:n//2])
plt.xlabel('freq (Hz)')
plt.ylabel('Amp')
plt.title(f'centroid_x fft. Max amp @{freq[np.argmax(fft_amp[:n//2])]}')
plt.grid(True)
plt.show()

In [ ]:
merged['centroid_delta_x'] = merged['centroid_x'] - merged['centroid_x'].mean()
merged['centroid_delta_y'] = merged['centroid_y'] - merged['centroid_y'].mean()
merged['centroid_delta'] = (merged['centroid_delta_x'] ** 2 + merged['centroid_delta_y'] ** 2) ** 0.5

merged['tm_delta_x'] = merged['tm_x'] - merged['tm_x'].mean()
merged['tm_delta_y'] = merged['tm_y'] - merged['tm_y'].mean()
merged['tm_delta'] = (merged['tm_delta_x'] ** 2 + merged['tm_delta_y'] ** 2) ** 0.5

merged

In [ ]:
# 傅里叶变换，分析质心x\y，快反镜x\y的抖动频谱



## wavefront change

In [ ]:
wf_df = pd.read_csv('data/20250611/20250611002/digitaloptical4Floor/波前数据/20250611 15：36：59(1-20%平顶31).TXT', delimiter='\t', header=0, encoding='gbk')
wf_df['时间戳'] = pd.to_datetime(wf_df['时间戳'], format='%Y-%m-%d-%H：%M：%S.%f')
wf_df.set_index('时间戳', inplace=True)

wf_df

In [ ]:
resampled_wf_df = wf_df.resample('0.01s').interpolate(method='time').dropna()
resampled_wf_df.resample('0.1s').wf_diff.plot()

In [ ]:
resampled_wf_df.wf_diff.diff().plot()

In [ ]:
from zernike import RZern

def fit_wavefront(zernike_coeffs, grid_size=256):
    """
    基于 Zernike 系数拟合波前

    参数:
    zernike_coeffs (list or np.ndarray): Zernike 系数列表
    grid_size (int): 生成波前网格的大小，默认为 256

    返回:
    np.ndarray: 拟合后的波前
    """
    n = len(zernike_coeffs)
    cart = RZern(n)
    rho = np.linspace(0, 1, grid_size)
    phi = np.linspace(0, 2 * np.pi, grid_size)
    RHO, PHI = np.meshgrid(rho, phi)
    cart.make_cart_grid(RHO, PHI)
    
    wavefront = np.zeros((grid_size, grid_size))
    for j in range(n):
        wavefront += zernike_coeffs[j] * cart.Zk[j]
    
    return wavefront

zernike_coeffs = np.concat(0, wf_df.iloc[0, 3:].values)
wavefront = fit_wavefront(zernike_coeffs)

In [ ]:
wf_df.iloc[0, 3:].values

wf_df = pd.read_csv('data/波前20250606 10：23：07(20%平顶31).TXT', delimiter='\t', header=0)
wf_df

## 功率数据

1. 时间等间距插值

In [ ]:
data = pd.read_csv('data/功率计数据/20250731 16：58：27(100%平顶18).TXT', encoding='gbk',sep='\t', header=None)
data.columns = ['time', 'power']
data.plot()

## 异常数据发现

## 大气环境处理

### load data

In [ ]:
import os
# 平均风速 平均温度 平均湿度 平均能见度 计算r0
os.listdir('data/20250611/大气数据/20250611_atmosphere')

In [ ]:
target_time ='''2025/6/6	10:24
2025/6/6	10:36
2025/6/6	10:45
2025/6/6	10:55
2025/6/6	11:26
2025/6/6	11:37
2025/6/6	14:29
2025/6/6	14:49
2025/6/9	11:42
2025/6/9	11:47
2025/6/9	13:26
2025/6/9	13:36
2025/6/9	13:46
2025/6/9	15:09
2025/6/9	15:18
2025/6/10	13:45
2025/6/10	13:52
2025/6/10	14:57
2025/6/10	15:06
2025/6/10	15:56
2025/6/10	16:02
2025/6/11	16:35
2025/6/11	16:47
2025/6/11	16:58
2025/6/12	9:04
2025/6/12	9:13
2025/6/12	9:17
2025/6/12	9:39
2025/6/12	9:42
2025/6/12	11:29
2025/6/12	11:35'''.split('\n')

target_time_df = pd.DataFrame(target_time, columns=['time'])
target_time_df['time'] = pd.to_datetime(target_time_df['time'], format="%Y/%m/%d\t%H:%M")

target_time_df


In [ ]:
def relocate(position, array):
    """
    按预设位置对分层参数进行加权平均（匹配实际大气分层位置）
    
    参数:
        array (list/np.ndarray): 待平均的分层参数数组（长度应等于layers）
        
    返回:
        np.ndarray: 平均后的参数数组（长度等于layers）
    """
    assert len(array) == len(position)
    
    ave = np.zeros_like(array)
    ave[0] = ((array[0] + array[1] + array[2] + array[3] + array[4]) * 20 + array[5] * 100) / 200
    # 后续层线性插值（根据position数组的位置间隔）
    for i in range(15):
        d = (i+2) * 200  # 当前层的位置（间隔200米）
        for j in range(15):
            # 找到d所在的position区间，进行线性插值
            if position[j] <= d <= position[j+1]:
                ave[i+1] = ((d - position[j]) * array[j] + (position[j+1] - d) * array[j+1]) / (position[j+1] - position[j])
            else:
                ave[i+1] = array[-1]  # 超出范围时取最后一个值
    return ave

In [ ]:
wind_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Wind\Wind.xlsx')
wind_df['time'] = pd.to_datetime('2025-'+wind_df['m/s'])
wind_df.drop('m/s', axis=1, inplace=True)
wind_df.info()

position = [int(p[:-1]) for p in wind_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

wind_df['avg'] = wind_df[[p for p in wind_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
wind_df['avg'].plot()

In [ ]:
temporature_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Temperature\Temperature.xlsx')
temporature_df['time'] = pd.to_datetime('2025-'+temporature_df['°C'])
temporature_df.drop('°C', axis=1, inplace=True)
temporature_df.info()

position = [int(p[:-1]) for p in temporature_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

temporature_df['avg'] = temporature_df[[p for p in temporature_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
temporature_df['avg'].plot()

# 光轴仿真分析

In [ ]:
from PIL import Image

pupil_img = np.asarray(Image.open('../doc/ao/closed-loop-pupil.bmp'))
px, py = get_mass_center(pupil_img)
axis_img = np.asarray(Image.open('../doc/ao/closed-loop-axis.bmp'))
ax, ay = get_mass_center(axis_img)

fig, [ax1, ax2] = plt.subplots(1,2)
ax1.imshow(pupil_img)
ax1.scatter(*get_mass_center(pupil_img), c='red')
ax2.imshow(axis_img)
ax2.scatter(*get_mass_center(axis_img), c='red')

In [ ]:
def shift_to_center_fft(image, cx, cy):
    """
    使用傅里叶移位将光斑移到图像中心（无插值，保全信息）
    
    参数:
        image: 2D array
        target_center: (cx, cy) 目标中心，默认为图像几何中心
    
    返回:
        shifted_image: 光斑已居中的图像
    """
    image = np.asarray(image, dtype=float)
    h, w = image.shape
    dx = w/2 - cx
    dy = h/2 - cy
    
    # 傅里叶移位：在频域乘以相位因子
    # 创建频率网格
    u = np.fft.fftfreq(w).reshape(1, -1)
    v = np.fft.fftfreq(h).reshape(-1, 1)

    # 相位因子：exp(-2πi (u*dx + v*dy))
    phase = np.exp(-2j * np.pi * (u * dx + v * dy))

    # 应用移位
    F = np.fft.fft2(image)
    F_shifted = F * phase
    shifted = np.real(np.fft.ifft2(F_shifted))

    # 保留非负强度（数值误差可能导致微小负值）
    shifted = np.clip(shifted, 0, None)
    return np.where(shifted < 1e-3, 0, shifted)

centered_pupil_img = shift_to_center_fft(pupil_img, px, py)

h,w = centered_pupil_img.shape
centered_pupil_img = centered_pupil_img[:, (w-h)//2:-(w-h)//2]

fig, [ax1, ax2] = plt.subplots(1,2)
ax1.imshow(pupil_img)
ax2.imshow(centered_pupil_img)

In [ ]:
from aotools.opticalpropagation import angularSpectrum, oneStepFresnel, twoStepFresnel

fig, [ax1, ax2, ax3] = plt.subplots(1,3)
plt.gray()

intensity_fft = np.abs(angularSpectrum(centered_pupil_img, 1064e-9, 5.6e-6, 5.6e-6, 3))
intensity_fresnel = np.abs(oneStepFresnel(centered_pupil_img, 1064e-9, 5.6e-6, 3))
intensity_2_fresnel = np.abs(twoStepFresnel(centered_pupil_img, 1064e-9, 5.6e-6, 5.6e-6*10, 3))

ax1.imshow(intensity_fft)
ax2.imshow(intensity_fresnel)
ax3.imshow(intensity_2_fresnel)

In [ ]:
fig, [ax1, ax2, ax3] = plt.subplots(1,3)
ax1.plot(intensity_fft[h//2, :])
ax2.plot(intensity_fresnel[h//2, :])
ax3.plot(intensity_2_fresnel[h//2, :])